In [1]:
import win32com.client as com
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import numpy as np
from functools import lru_cache
from collections import defaultdict
from shapely.geometry import LineString
from shapely import wkt
import seaborn as sns

In [3]:
folder = r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara'

In [2]:
#Red base GDL (sin links agregados por Johan y Daniel)
original_ver = r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Visum Projects\Jeannette\RedOSMNX_AMG_24 - Jul.ver'

import win32com.client
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(original_ver)
C = win32com.client.constants

## Open Sebastian shp Visum Links with attributes

In [4]:
links_with_atts = gpd.read_file(os.path.join(folder, 'VisumLinks With TransCAD Atts', 'merge_visum_osm_sebastian', 'red_visum_atributos.shp'))
links_with_atts

,No,FromNodeNo,ToNodeNo,TypeNo,TSysSet,Length,U,V,KEY,HIGHWAY,cap_final,carr_final,velprom_fi,vel_final,con_osm,geometry
0,1,1,103492,0,"B,C",4.827825,267537966,7306651630,0,motorway,4000.0,2.0,24.000000,30.0,1,"LINESTRING (-103.2463 20.6161, -103.2473 20.61..."
1,1,103492,1,0,None,0.000000,0,0,0,None,0.0,0.0,0.000000,0.0,0,"LINESTRING (-103.2905 20.6279, -103.2894 20.62..."
2,2,1,15708,0,"B,C",0.146029,267537966,5837556433,0,motorway_link,4000.0,2.0,23.999956,30.0,1,"LINESTRING (-103.2463 20.6161, -103.2465 20.61..."
3,2,15708,1,0,None,0.000000,0,0,0,None,0.0,0.0,0.000000,0.0,0,"LINESTRING (-103.2477 20.6163, -103.2475 20.61..."
4,3,2,3,0,"B,C",0.520237,267538751,273140976,0,motorway,2500.0,1.0,53.999923,80.0,1,"LINESTRING (-103.1364 20.6056, -103.1343 20.60..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593963,500924,213454,205191,0,"B,C,W",0.010734,13675767594,13675767594,1,service,2000.0,2.0,18.500206,30.0,1,"LINESTRING (-103.342 20.6756, -103.342 20.6756)"
593964,500925,206297,213455,0,None,0.000000,0,0,0,None,0.0,0.0,0.000000,0.0,0,"LINESTRING (-103.3527 20.7297, -103.3528 20.7296)"
593965,500925,213455,206297,0,TL,0.010452,4589568757,7834105323,1,None,0.0,0.0,0.000000,0.0,0,"LINESTRING (-103.3528 20.7296, -103.3527 20.7297)"
593966,500926,206297,213456,0,None,0.000000,0,0,0,None,0.0,0.0,0.000000,0.0,0,"LINESTRING (-103.3527 20.7297, -103.3527 20.7298)"


In [11]:
links_with_atts['carr_final'].value_counts()

carr_final
2.0     327915
0.0     136827
1.0      69893
3.0      26931
4.0      19393
6.0       9100
8.0       1720
5.0        974
1.5        735
3.5        252
2.5        101
7.0         82
4.5         40
12.0         5
Name: count, dtype: int64

In [12]:
visum_links = Visum.Net.Links

links =  links_with_atts.copy()
links = links.reset_index(drop=True)
links.index = links.index + 1

capacidad = list(zip(
    links.index,
    links['cap_final'].astype(int)
))

carriles = list(zip(
    links.index,
    links['carr_final']
))

velprom = list(zip(
    links.index,
    links['velprom_fi']
))

limvel = list(zip(
    links.index,
    links['vel_final']
))

visum_links.SetMultiAttValues("CAPACIDAD_FINAL", capacidad)
visum_links.SetMultiAttValues("CARRILES_FINAL", carriles)
visum_links.SetMultiAttValues("VELPROM_FINAL", velprom)
visum_links.SetMultiAttValues("LIMVEL_FINAL", limvel)